# Relatório de Óbitos — Município de Araranguá/SC (2024–2025)
### Fonte: SIM (Sistema de Informação sobre Mortalidade) — Ministério da Saúde

Este notebook documenta, passo a passo, o processo de coleta, tratamento e
análise dos dados de óbitos do município de **Araranguá/SC**, com foco no
período mais recente disponível: **2024 e 2025**.




## Etapa 1 — Configuração e autenticação na API

A API Dados Abertos da Saúde exige um token de autenticação enviado no
cabeçalho (`header`) de cada requisição. Para não expor essa chave
diretamente no código, ela é armazenada nos **Secrets do Google Colab**
(ícone de chave na barra lateral) e recuperada em tempo de execução com
`userdata.get('API_KEY')`.

Isso evita que a chave de API fique visível no notebook ou seja enviada
acidentalmente para o GitHub/GitLab caso o notebook seja compartilhado.


In [ ]:
import pandas as pd
import requests
from google.colab import userdata

# A chave de API fica guardada nos "Secrets" do Colab, nunca escrita no código
API_KEY = userdata.get('API_KEY')
headers = {'Authorization': f'Bearer {API_KEY}'}
base_url = 'https://apidadosabertos.saude.gov.br/vigilancia-e-meio-ambiente/sistema-de-informacao-sobre-mortalidade'


## Etapa 2 — Definição dos parâmetros de busca

Antes de consultar a API, definimos:

- **CAMPOS**: os campos do SIM que interessam para esta análise (data do
  óbito, idade, sexo, raça/cor, escolaridade, município de residência, local
  de ocorrência e causa básica do óbito).
- **COD_ARARANGUA**: o código IBGE (6 dígitos) do município de Araranguá/SC,
  usado depois para filtrar os registros.
- **ANO_INICIO / ANO_FIM**: o período de interesse da análise — **2024 a
  2025**. Para analisar outro intervalo, basta alterar esses dois valores.
- **LIMIT / MAX_PAGINAS**: parâmetros de paginação. A API retorna os
  registros em blocos (páginas), então é preciso percorrer várias páginas até
  esgotar os dados disponíveis.


In [ ]:
CAMPOS = ["dtobito", "idade", "sexo", "racacor", "esc2010", "codmunres", "lococor", "causabas"]
COD_ARARANGUA = "420140"  # codigo IBGE (6 digitos) do municipio de Araranguá/SC
ANO_INICIO, ANO_FIM = 2024, 2025  # periodo de analise deste relatorio

LIMIT = 1000
MAX_PAGINAS = 800  # cada pagina = 100 registros


## Etapa 3 — Coleta paginada dos dados da API

A API não retorna todos os registros de uma vez, então o código percorre as
páginas em um laço `for`, avançando o `offset` a cada iteração
(`offset += LIMIT`).

A cada página, os registros retornados (`dados`) são acumulados na lista
`todos_registros`. Quando a API retorna uma página vazia, isso significa que
não há mais dados a buscar, e o laço é interrompido (`break`).

O `print` a cada iteração serve para acompanhar o progresso da coleta em
tempo real (quantos registros já foram baixados até aquele ponto).

Ao final, a lista de registros é convertida em um DataFrame do pandas,
mantendo apenas os campos definidos em `CAMPOS`, com os nomes das colunas em
maiúsculo para padronização.


In [ ]:
todos_registros = []
offset = 0

for pagina in range(MAX_PAGINAS):
    params = {'limit': LIMIT, 'offset': offset}
    resp = requests.get(base_url, headers=headers, params=params)
    resp.raise_for_status()
    dados = resp.json().get('sim', [])

    if not dados:
        print(f"Sem mais dados a partir do offset {offset}. Encerrando busca.")
        break

    todos_registros.extend(dados)
    offset += LIMIT
    print(f"Pagina {pagina + 1}: offset {offset - LIMIT}, {len(dados)} registros (total acumulado: {len(todos_registros)})")

df = pd.DataFrame(todos_registros)
df = df[CAMPOS]
df.columns = [c.upper() for c in df.columns]


## Etapa 4 — Decodificação dos campos conforme o Dicionário de Dados do SIM

Os dados brutos do SIM vêm codificados numericamente (ex.: sexo = `"1"` ou
`"2"`). Para tornar a análise legível, foram criadas funções e dicionários de
mapeamento com base no dicionário de dados oficial:

- **`decodifica_idade`**: a idade no SIM vem em um formato especial de 3
  dígitos, onde o 1º dígito indica a unidade (minuto, hora, mês, ano) e os 2
  últimos a quantidade. Ex.: `"467"` = 67 anos (`4` = "ano(s)").
- **`decodifica_dtobito`**: converte a data no formato `ddmmaaaa` para o
  formato `dd/mm/aaaa`, mais legível.
- **`MAPA_SEXO`, `MAPA_RACACOR`, `MAPA_ESC2010`, `MAPA_LOCOCOR`**: dicionários
  que traduzem os códigos numéricos para as descrições textuais
  correspondentes.
- **`decodifica_causabas`**: diferente da versão anterior deste relatório,
  agora a causa básica do óbito (`CAUSABAS`, um código CID-10) também é
  decodificada. A tabela oficial de categorias da CID-10 é baixada
  diretamente do repositório público do DataSUS no GitHub, e o mapeamento é
  feito pelos 3 primeiros caracteres do código (ex.: `"J449"` → categoria
  `"J44"` → "Outras doenças pulmonares obstrutivas crônicas").


In [ ]:
def decodifica_idade(valor):
    """IDADE = 1 digito de unidade + 2 digitos de quantidade.
    1-minuto, 2-hora, 3-mes, 4-ano, 5-mais de 100 anos, 9-ignorado."""
    if pd.isna(valor) or valor in (None, ""):
        return None
    v = str(valor).zfill(3)
    unidade, qtd = v[0], v[1:]
    mapa_unidade = {
        "1": "minuto(s)",
        "2": "hora(s)",
        "3": "mes(es)",
        "4": "ano(s)",
        "5": "ano(s) (100+)",
        "9": "idade ignorada",
    }
    if unidade == "9":
        return "Ignorado"
    return f"{int(qtd)} {mapa_unidade.get(unidade, 'unidade desconhecida')}"

def decodifica_dtobito(valor):
    """DTOBITO no padrao ddmmaaaa (as vezes sem zero a esquerda)."""
    if pd.isna(valor) or valor in (None, ""):
        return None
    v = str(valor).zfill(8)
    try:
        return pd.to_datetime(v, format="%d%m%Y").strftime("%d/%m/%Y")
    except ValueError:
        return None

MAPA_SEXO = {"1": "Masculino", "2": "Feminino", "0": "Ignorado", "9": "Ignorado",
             "M": "Masculino", "F": "Feminino", "I": "Ignorado"}

MAPA_RACACOR = {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indigena"}

MAPA_ESC2010 = {
    "0": "Sem escolaridade",
    "1": "Fundamental I (1a a 4a serie)",
    "2": "Fundamental II (5a a 8a serie)",
    "3": "Medio (antigo 2o Grau)",
    "4": "Superior incompleto",
    "5": "Superior completo",
    "9": "Ignorado",
}

MAPA_LOCOCOR = {
    "1": "Hospital",
    "2": "Outros estabelecimentos de saude",
    "3": "Domicilio",
    "4": "Via publica",
    "5": "Outros",
    "6": "Aldeia indigena",
    "9": "Ignorado",
}

# Tabela oficial de categorias da CID-10 (usada para decodificar CAUSABAS)
!wget -q -O CID-10-CATEGORIAS.CSV "https://raw.githubusercontent.com/cleytonferrari/CidDataSus/master/CIDImport/Repositorio/Resources/CID-10-CATEGORIAS.CSV"
cid10 = pd.read_csv("CID-10-CATEGORIAS.CSV", sep=";", encoding="latin1")
cid10["CAT"] = cid10["CAT"].astype(str).str.strip().str.upper()
MAPA_CID10 = dict(zip(cid10["CAT"], cid10["DESCRICAO"]))

def decodifica_causabas(codigo):
    if pd.isna(codigo) or str(codigo).strip() == "":
        return None
    cod = str(codigo).strip().upper()
    return MAPA_CID10.get(cod[:3], f"Desconhecido ({cod})")

df["DTOBITO"] = df["DTOBITO"].apply(decodifica_dtobito)
df["IDADE"] = df["IDADE"].apply(decodifica_idade)
df["SEXO"] = df["SEXO"].map(MAPA_SEXO).fillna(df["SEXO"])
df["RACACOR"] = df["RACACOR"].map(MAPA_RACACOR).fillna(df["RACACOR"])
df["ESC2010"] = df["ESC2010"].map(MAPA_ESC2010).fillna(df["ESC2010"])
df["LOCOCOR"] = df["LOCOCOR"].map(MAPA_LOCOCOR).fillna(df["LOCOCOR"])
df["CAUSABAS_DESC"] = df["CAUSABAS"].apply(decodifica_causabas)


## Etapa 5 — Filtragem dos registros de Araranguá

Com os campos já decodificados, o passo seguinte é isolar apenas os óbitos de
residentes em Araranguá. Como `CODMUNRES` pode vir sem o zero à esquerda
(ex.: `"20140"` em vez de `"420140"`), aplicamos `.zfill(6)` antes da
comparação, para garantir que o código sempre tenha 6 dígitos e a comparação
seja feita corretamente.

O resultado é o DataFrame `df_ararangua`, com o índice reiniciado
(`reset_index(drop=True)`).


In [ ]:
df_ararangua = df[df["CODMUNRES"].astype(str).str.zfill(6) == COD_ARARANGUA].reset_index(drop=True)

print(f"Total de registros verificados: {len(df)}")
print(f"Registros de Araranguá encontrados: {len(df_ararangua)}")
df_ararangua


## Etapa 6 — Fonte de dados rápida e reprodutível (com CID-10 e recorte 2024–2025)

A coleta paginada via API (Etapas 1 a 5) pode levar bastante tempo — centenas
de páginas — e depende de uma chave de API pessoal. Nesta etapa, usamos uma
alternativa mais rápida: uma base pública, já consolidada, com **todos os
óbitos de Santa Catarina**, disponibilizada via Google Drive.

Diferente da versão anterior deste relatório (que lia um CSV **já filtrado e
salvo previamente** no Drive, mas com a causa do óbito ainda em código), esta
versão:

1. Lê a base completa de SC (~1,5 milhão de registros) **em blocos**
   (`chunksize=200_000`), para não sobrecarregar a memória;
2. Filtra apenas os registros de **Araranguá** a cada bloco;
3. Decodifica idade, escolaridade e a **causa básica do óbito** (via tabela
   CID-10, baixada automaticamente do DataSUS/GitHub — o mesmo mapeamento da
   Etapa 4);
4. Converte a data para `dd/mm/aaaa`;
5. Restringe o resultado final ao período de análise deste relatório —
   **`ANO_INICIO` a `ANO_FIM` (2024–2025)**, definido na Etapa 2.

O resultado sobrescreve `df_ararangua`, agora já **decodificado e recortado
para o período de interesse** — pronto para as próximas etapas.


In [ ]:
!pip install gdown -q

import gdown
import pandas as pd

file_id = '1MxWxvZVucgj8ZxFSl3p8N2PscCqQBw-i'
output = 'sim_sc_processado.csv'
gdown.download(id=file_id, output=output, quiet=False)

CAMPOS_UP = [c.upper() for c in CAMPOS]

# Leitura em blocos (a base tem ~1,5 milhão de linhas) já filtrando Araranguá
blocos = []
total_lido = 0
for bloco in pd.read_csv(output, sep=",", encoding="utf-8-sig", usecols=CAMPOS_UP,
                          dtype=str, chunksize=200_000):
    total_lido += len(bloco)
    blocos.append(bloco[bloco["CODMUNRES"].astype(str).str[:6] == COD_ARARANGUA])
df_ararangua = pd.concat(blocos, ignore_index=True)
del blocos

df_ararangua["DTOBITO"] = pd.to_datetime(df_ararangua["DTOBITO"], errors="coerce").dt.strftime("%d/%m/%Y")
df_ararangua["IDADE"] = df_ararangua["IDADE"].apply(decodifica_idade)
df_ararangua["ESC2010"] = df_ararangua["ESC2010"].map(MAPA_ESC2010).fillna(df_ararangua["ESC2010"])
df_ararangua["CAUSABAS_DESC"] = df_ararangua["CAUSABAS"].apply(decodifica_causabas)

print(f"Total de registros verificados: {total_lido}")
print(f"Registros de Araranguá (todos os anos): {len(df_ararangua)}")

# Recorte para o período de análise (2024-2025)
ano_obito = pd.to_datetime(df_ararangua["DTOBITO"], format="%d/%m/%Y", errors="coerce").dt.year
df_ararangua = df_ararangua[ano_obito.between(ANO_INICIO, ANO_FIM)].reset_index(drop=True)

print(f"Registros de Araranguá em {ANO_INICIO}-{ANO_FIM}: {len(df_ararangua)}")
df_ararangua


## Etapa 7 — Preparação de variáveis derivadas para análise

Para viabilizar as análises e gráficos, algumas colunas auxiliares foram
criadas a partir dos dados já decodificados (o mesmo procedimento se aplica
independentemente de `df_ararangua` ter vindo da Etapa 5 ou da Etapa 6, já
que ambas produzem os campos no mesmo formato):

- **`DTOBITO_DT`**: conversão da data (texto `dd/mm/aaaa`) para o tipo
  `datetime` do pandas, permitindo operações temporais.
- **`ANOMES`**: extrai o período (ano-mês) de cada óbito, usado para agrupar
  e visualizar a série temporal de óbitos por mês.
- **`IDADE_ANOS`**: a função `idade_em_anos` converte o texto da idade (ex.:
  `"60 ano(s)"`) para um número inteiro. Idades registradas em minutos,
  horas ou meses são tratadas como menores que 1 ano (valor `0`).
- **`FAIXA_ETARIA`**: usando `pd.cut`, a idade em anos é agrupada em faixas
  etárias (`<1`, `1-9`, `10-19`, ..., `90+`), facilitando a leitura do
  perfil etário dos óbitos.


In [ ]:
df = df_ararangua

df["DTOBITO_DT"] = pd.to_datetime(df["DTOBITO"], format="%d/%m/%Y")
df["ANOMES"] = df["DTOBITO_DT"].dt.to_period("M").astype(str)

def idade_em_anos(valor):
    """Converte '60 ano(s)' -> 60. Para meses/horas/minutos, considera menor
    que 1 ano (retorna 0)."""
    if "ano(s)" in str(valor):
        try:
            return int(str(valor).split(" ")[0])
        except ValueError:
            return None
    return 0

df["IDADE_ANOS"] = df["IDADE"].apply(idade_em_anos)

faixas = [0, 1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 200]
rotulos_faixas = ["<1", "1-9", "10-19", "20-29", "30-39", "40-49",
                   "50-59", "60-69", "70-79", "80-89", "90+"]
df["FAIXA_ETARIA"] = pd.cut(df["IDADE_ANOS"], bins=faixas, labels=rotulos_faixas, right=False)


## Etapa 8 — Análise exploratória e visualizações (2024–2025)

Com as variáveis derivadas prontas, geramos um conjunto de gráficos para
descrever o perfil da mortalidade em Araranguá **no período de 2024 a 2025**
sob diferentes ângulos:

1. **Óbitos por mês** — série temporal, para identificar tendências ou picos
   sazonais dentro do período.
2. **Óbitos por faixa etária** — mostra em quais grupos etários a
   mortalidade é maior.
3. **Óbitos por sexo** — distribuição entre masculino e feminino.
4. **Óbitos por raça/cor** — distribuição segundo a variável raça/cor.
5. **Óbitos por escolaridade** — perfil educacional dos óbitos registrados.
6. **Óbitos por local de ocorrência** — hospital, domicílio, via pública,
   etc.
7. **Principais causas de óbito** — top 10 causas básicas mais frequentes
   (coluna `CAUSABAS_DESC`, agora já decodificada via CID-10 na Etapa 4/6 —
   diferente da versão anterior, que deixava esse campo pendente).

Cada gráfico usa `matplotlib` e segue o mesmo padrão: título, rótulos dos
eixos e `tight_layout()` para evitar cortes na exibição.


In [ ]:
import matplotlib.pyplot as plt

# 1) Serie temporal: obitos por mes
obitos_por_mes = df.groupby("ANOMES").size()
print(f"Total de óbitos no período {ANO_INICIO}-{ANO_FIM}: {len(df)}")

plt.figure(figsize=(10, 4))
plt.plot(obitos_por_mes.index, obitos_por_mes.values, marker="o")
plt.title(f"Óbitos por mês — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.xlabel("Mês")
plt.ylabel("Número de óbitos")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 2) Obitos por faixa etaria
obitos_por_faixa = df["FAIXA_ETARIA"].value_counts().reindex(rotulos_faixas)
plt.figure(figsize=(10, 4))
plt.bar(obitos_por_faixa.index.astype(str), obitos_por_faixa.values)
plt.title(f"Óbitos por faixa etária — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.xlabel("Faixa etária")
plt.ylabel("Número de óbitos")
plt.tight_layout()
plt.show()

# 3) Obitos por sexo
obitos_por_sexo = df["SEXO"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(obitos_por_sexo.index, obitos_por_sexo.values)
plt.title(f"Óbitos por sexo — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.ylabel("Número de óbitos")
plt.tight_layout()
plt.show()

# 4) Obitos por raca/cor
obitos_por_raca = df["RACACOR"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(obitos_por_raca.index, obitos_por_raca.values)
plt.title(f"Óbitos por raça/cor — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.ylabel("Número de óbitos")
plt.tight_layout()
plt.show()

# 5) Obitos por escolaridade
obitos_por_esc = df["ESC2010"].value_counts()
plt.figure(figsize=(10, 4))
plt.bar(obitos_por_esc.index, obitos_por_esc.values)
plt.title(f"Óbitos por escolaridade — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.ylabel("Número de óbitos")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 6) Obitos por local de ocorrencia
obitos_por_local = df["LOCOCOR"].value_counts()
plt.figure(figsize=(8, 4))
plt.barh(obitos_por_local.index, obitos_por_local.values)
plt.title(f"Óbitos por local de ocorrência — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.xlabel("Número de óbitos")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 7) Principais causas de obito (top 10)
top_causas = df["CAUSABAS_DESC"].value_counts().head(10)
plt.figure(figsize=(10, 6))
plt.barh(top_causas.index, top_causas.values)
plt.title(f"Principais causas de óbito — Araranguá, {ANO_INICIO}-{ANO_FIM}")
plt.xlabel("Número de óbitos")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
